In [1]:
import botometer
import pandas as pd

rapidapi_key ="01b2292ddcmsh077e491a5e45c25p10a295jsn8b171a0fcace"
    #"97027577damsha428ea64d3ea3b8p1f094cjsne615bf13f735"
    #"5a0e1ee701msh1359f50de176cb1p15ea21jsna217b4177753"
    #"7debb6de54msh03588221ff43119p1987d6jsn16948158e014"
    #"b6d7bb44d0msh492ca3dfda6bca6p1cac23jsnf7b3bf47623b"
    #"3e4314cc81msh3eae724c8979044p1247a5jsndfd06cf147b6"
    #"39edccb09bmshe41b78cd82637d5p1fc120jsn80ba24dc07e6"
    #"0bd9a74c33msh4f5d0b914b1365ep1e5687jsnb04b542dce12"
    #"749c962788mshf2573eb6d1021f9p171bb6jsn49d981cc6759"
    #"234408d8ddmshabf2780f07924efp194665jsndc05764c7c1f"
    #"abba500b83mshcda140c16d64b10p180ee1jsn8e3923755e28"
    #"fc93877f79msh43fe708ddb46bacp144982jsn3f2e5b1f4fed"
    #"1d30dd2683msh1f74144abd23bf3p100e9fjsn0a92a8403692"
    #"ba61e85719msh55e1973851e7825p17bc51jsn99105fffbe2b"

bomx = botometer.BotometerX(rapidapi_key=rapidapi_key)

df_users = pd.read_csv("df_users_gossip.csv")

E:\SEM5\Challenge\env\CGML\Lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [2]:
import json
import math
import os
from time import sleep

# Your user IDs
user_ids = df_users["twitter_id"].tolist()
batch_size = 100
max_calls_per_hour = 50

# File where progress is stored
save_path = "scores_gos.json"

# --------------------------------------------------
# Load existing progress safely
# --------------------------------------------------
if os.path.exists(save_path):
    with open(save_path, "r") as f:
        try:
            saved_scores = json.load(f)
            # Handle double-encoded JSON
            if isinstance(saved_scores, str):
                saved_scores = json.loads(saved_scores)
        except json.JSONDecodeError:
            print("Warning: scores_gos.json is corrupted or empty — starting fresh.")
            saved_scores = []
else:
    saved_scores = []

# Ensure it's a list
if not isinstance(saved_scores, list):
    print(f"Warning: Expected list but got {type(saved_scores)}. Resetting save file.")
    saved_scores = []

# --------------------------------------------------
# Determine already processed IDs
# --------------------------------------------------
processed_ids = {entry["user_id"] for entry in saved_scores}
remaining_ids = [uid for uid in user_ids if str(uid) not in processed_ids]

print(f"Resuming: {len(processed_ids)} already done, {len(remaining_ids)} remaining.")

Resuming: 31442 already done, 1228 remaining.


In [47]:
# --------------------------------------------------
# Process in batches with saving + rate limit
# --------------------------------------------------
calls_this_hour = 0
for i in range(0, len(remaining_ids), batch_size):
    if calls_this_hour >= max_calls_per_hour:
        print("Reached API limit of 1000 calls per hour. Stopping for now.")
        break

    batch = remaining_ids[i:i + batch_size]
    batch_num = len(processed_ids) // batch_size + (i // batch_size) + 1
    total_batches = math.ceil(len(user_ids) / batch_size)
    print(f"Processing batch {batch_num} / {total_batches} (IDs {i}–{i+len(batch)-1})")

    try:
        scores = bomx.get_botscores_in_batch(user_ids=batch)

        # Ensure scores is a list of dicts
        if isinstance(scores, str):
            scores = json.loads(scores)

        if not isinstance(scores, list):
            print(f"Unexpected response type: {type(scores)} — skipping batch.")
            continue

        saved_scores.extend(scores)
        processed_ids.update([s.get("user_id") for s in scores if isinstance(s, dict)])
        calls_this_hour += 1

        # Save progress after each batch
        with open(save_path, "w") as f:
            json.dump(saved_scores, f, indent=2)

        print(f"Saved {len(saved_scores)} total results so far.")

    except Exception as e:
        print(f"Error on batch {batch_num}: {e}")
        with open(save_path, "w") as f:
            json.dump(saved_scores, f, indent=2)
        break

    sleep(1)

print(f"Finished run. {len(saved_scores)} total scores saved.")


Processing batch 314 / 327 (IDs 0–99)
Saved 31393 total results so far.
Processing batch 315 / 327 (IDs 100–199)
Saved 31393 total results so far.
Processing batch 316 / 327 (IDs 200–299)
Saved 31393 total results so far.
Processing batch 317 / 327 (IDs 300–399)
Saved 31393 total results so far.
Processing batch 318 / 327 (IDs 400–499)
Saved 31393 total results so far.
Processing batch 319 / 327 (IDs 500–599)
Saved 31393 total results so far.
Processing batch 320 / 327 (IDs 600–699)
Saved 31393 total results so far.
Processing batch 321 / 327 (IDs 700–799)
Saved 31393 total results so far.
Processing batch 322 / 327 (IDs 800–899)
Saved 31393 total results so far.
Processing batch 323 / 327 (IDs 900–999)
Saved 31393 total results so far.
Processing batch 324 / 327 (IDs 1000–1099)
Saved 31393 total results so far.
Processing batch 325 / 327 (IDs 1100–1199)
Saved 31393 total results so far.
Processing batch 326 / 327 (IDs 1200–1276)
Saved 31442 total results so far.
Finished run. 31442 to

In [3]:
import json
import pandas as pd

# Load the JSON file
with open('scores_gos.json', 'r') as f:
    data = json.load(f)

n_entries = sum(1 for entry in data)

df_users = pd.read_csv("df_users_gossip.csv")
print(f"Entries {n_entries}")
print(f"Entries df_users {len(df_users['twitter_id'].unique())}")

Entries 31442
Entries df_users 32670


In [6]:
import pandas as pd

with open("scores_gos.json", "r") as f:
    user_data = json.load(f)  # gives you a string

df_json = pd.DataFrame(user_data)

df_csv = pd.read_csv("df_users_gossip.csv")
df_csv["twitter_id"] = df_csv["twitter_id"].astype(str)
df_json["user_id"] = df_json["user_id"].astype(str)
# Merge (left join keeps all CSV rows, fill from JSON)
merged_df = df_csv.merge(df_json, how="left", left_on="twitter_id", right_on="user_id")

# Optionally drop duplicate key columns if both exist
if "user_id" in merged_df.columns:
    merged_df = merged_df.drop(columns=["user_id"])

# Overwrite the original CSV
merged_df.to_csv("df_users_gossip.csv", index=False)

print("CSV updated successfully.")


CSV updated successfully.


In [8]:
print(df_json.head())

   bot_score                      timestamp   user_id         username
0       0.03  Thu, 01 Jun 2023 00:20:10 GMT   2883841            enews
1       0.65  Thu, 01 Jun 2023 01:58:13 GMT  10409622     thedextazlab
2       0.03  Thu, 01 Jun 2023 11:20:00 GMT  12870772  birmingham_live
3       0.04  Wed, 14 Apr 2021 15:58:39 GMT  18125878   patricktoneill
4       0.59  Wed, 17 Oct 2018 01:27:06 GMT  20587567  christianebuddy


In [9]:
print(df_json.shape)
print(df_json.head())

(31442, 4)
   bot_score                      timestamp   user_id         username
0       0.03  Thu, 01 Jun 2023 00:20:10 GMT   2883841            enews
1       0.65  Thu, 01 Jun 2023 01:58:13 GMT  10409622     thedextazlab
2       0.03  Thu, 01 Jun 2023 11:20:00 GMT  12870772  birmingham_live
3       0.04  Wed, 14 Apr 2021 15:58:39 GMT  18125878   patricktoneill
4       0.59  Wed, 17 Oct 2018 01:27:06 GMT  20587567  christianebuddy
